In [ ]:
from pathlib import Path
import json
import re
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch

PROJECT_ROOT = Path(".").resolve()
BASELINE_DIR = PROJECT_ROOT / "output" / "laser_injection" / "baseline"
CARD_OWNED_DIR = PROJECT_ROOT / "output" / "laser_injection" / "card_owned"
CARD_OWNED_CONS_DIR = PROJECT_ROOT / "output" / "cardinality_consistency" / "card_owned"
BASELINE_TAG = "ontology"
SELECT_MODEL_TAG = "card_owned"

MODEL_SOURCES = {
    "baseline": {"dir": BASELINE_DIR, "tag": BASELINE_TAG, "label": "Baseline"},
    "card_owned": {
        "dir": CARD_OWNED_DIR,
        "tag": "ontology",
        "label": "card_owned",
        "cons_dir": CARD_OWNED_CONS_DIR,
    },
}

EXAMPLE_ATTACK_CASES = [
    "red_laser_on_green_top",
    "ir_laser_on_green_top",
    "ir_laser_on_green_middle",
    "green_laser_on_red_bottom",
    "ir_laser_on_red_middle",
    "ir_laser_on_red_bottom",
    "red_laser_on_yellow_top",
    "green_laser_on_yellow_bottom",
    "ir_laser_on_yellow_top",
    "ir_laser_on_yellow_bottom",
    "red_laser_on_off_top",
    "green_laser_on_off_bottom",
    "ir_laser_on_off_top",
    "ir_laser_on_off_middle",
    "ir_laser_on_off_bottom",
    "green_laser_on_red_yellow_bottom",
    "ir_laser_on_red_yellow_bottom",
]

BENIGN_DISPLAY = {
    "red": "red",
    "green": "green",
    "yellow": "yellow",
    "red_yellow": "red+yellow",
    "off": "off",
}
PLACEMENT_DISPLAY = {
    "on_top": "on top",
    "opposite": "opposite",
    "top": "top",
    "middle": "middle",
    "bottom": "bottom",
}
LASER_COLORS = ["green", "red", "ir"]
WRONG_STATE_SHORT = {
    "red": "R",
    "green": "G",
    "yellow": "Y",
    "red_yellow": "R+Y",
    "off": "off",
}
_COMBO_RE = re.compile(
    r"^(?P<laser>red|green|ir)_laser_on_"
    r"(?P<benign>.+?)_"
    r"(?P<placement>on_top|opposite|top|middle|bottom)$"
)

DISPLAY_TITLES = {
    "green": "Green Laser Injection",
    "red": "Red Laser Injection",
    "ir": "IR Light Injection",
}
TEXT_SCALE = 1.0
TITLE_SIZE = 12 * TEXT_SCALE
SUPTITLE_SIZE = 14 * TEXT_SCALE
AXIS_LABEL_SIZE = 11 * TEXT_SCALE
TICK_LABEL_SIZE = 10 * TEXT_SCALE
LEGEND_SIZE = 8 * TEXT_SCALE
ANNOT_SIZE = 8 * TEXT_SCALE


def parse_combo_key(key):
    if key is None or (isinstance(key, float) and pd.isna(key)):
        return None
    m = _COMBO_RE.match(str(key))
    return m.groupdict() if m else None


def label_attack_axis(key) -> str:
    parsed = parse_combo_key(key)
    if not parsed:
        return str(key)
    benign = BENIGN_DISPLAY.get(parsed["benign"], parsed["benign"])
    if parsed["laser"] in {"red", "green"}:
        return f"{benign} benign"
    pl = PLACEMENT_DISPLAY.get(parsed["placement"], parsed["placement"])
    return f"{benign} ({pl})"


def unique_example_cases(cases=None) -> list[str]:
    out, seen = [], set()
    for k in (cases if cases is not None else EXAMPLE_ATTACK_CASES):
        s = str(k).strip()
        if not s or s in seen:
            continue
        seen.add(s)
        out.append(s)
    return out


def group_cases_by_laser(cases: list[str]) -> dict[str, list[str]]:
    groups = {laser: [] for laser in LASER_COLORS}
    for key in cases:
        parsed = parse_combo_key(key)
        laser = parsed["laser"] if parsed else "other"
        groups.setdefault(laser, []).append(key)
    return {k: v for k, v in groups.items() if v}


EXAMPLE_ATTACK_CASES = unique_example_cases(EXAMPLE_ATTACK_CASES)


In [ ]:
if "baseline" not in MODEL_SOURCES:
    raise ValueError("MODEL_SOURCES must include 'baseline'.")
if SELECT_MODEL_TAG not in MODEL_SOURCES:
    raise KeyError(f"SELECT_MODEL_TAG={SELECT_MODEL_TAG!r} not in MODEL_SOURCES")

other_alias = SELECT_MODEL_TAG
other_spec = MODEL_SOURCES[other_alias]

TWO_ALIASES = ["baseline", other_alias]
TWO_COLORS = {
    "baseline": "#1f4e79",
    other_alias: "#d62728",
}


def _combo_pattern_from_path(path: str):
    if "/generated/" not in path:
        return None, None
    seg = path.split("/generated/", 1)[1].split("/")
    combo = seg[0]
    if combo == "benign_crops" or "_laser_on_" not in combo:
        return None, None
    pattern = seg[1] if len(seg) > 1 else "pattern_?"
    return combo, pattern


def _stream_combined_with_majority(details_path: Path, combo_filter: set[str]):
    per_pat = defaultdict(lambda: [0, 0, 0])
    per_combo = defaultdict(lambda: [0, 0, 0])
    wrong_counts = defaultdict(lambda: defaultdict(int))
    n_used = 0

    with open(details_path, encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            combo, pattern = _combo_pattern_from_path(str(row.get("path") or ""))
            if combo is None or combo not in combo_filter:
                continue

            gt = str(row.get("gt_state") or "")
            pred = str(row.get("pred_state") or "")
            flip = bool(gt and pred and gt != pred)
            fooled = not bool(row.get("invalid", not bool(row.get("card_ok"))))
            combined = flip and fooled
            n_used += 1

            for acc in (per_pat[(combo, pattern)], per_combo[combo]):
                acc[0] += 1
                acc[1] += int(combined)
                acc[2] += int(flip)

            if combined and pred:
                wrong_counts[(combo, pattern)][pred] += 1

    return per_pat, per_combo, wrong_counts, n_used


def _baseline_stats_from_ranking(combo_filter: set[str]):
    src_dir = Path(MODEL_SOURCES["baseline"]["dir"])
    path = src_dir / "attack_ranking.json"
    if not path.is_file():
        return None
    report = json.loads(path.read_text())
    block = (
        report.get("baseline")
        or report.get(MODEL_SOURCES["baseline"].get("tag", "baseline"))
        or {}
    )
    per_combo = defaultdict(lambda: [0, 0, 0])
    per_pat = defaultdict(lambda: [0, 0, 0])

    for entry in block.get("attack_ranking") or []:
        combo = entry.get("combo_key")
        if combo not in combo_filter:
            continue
        n = int(entry.get("total") or 0)
        succ = int(
            entry.get("attack_successes")
            or round(float(entry.get("attack_success_rate") or 0) * n)
        )
        per_combo[combo] = [n, succ, succ]

    for entry in block.get("pattern_attack_ranking") or []:
        combo = entry.get("combo_key")
        if combo not in combo_filter:
            continue
        pat = str(entry.get("pattern_id", "pattern_?"))
        n = int(entry.get("total") or 0)
        succ = int(
            entry.get("attack_successes")
            or round(float(entry.get("attack_success_rate") or 0) * n)
        )
        per_pat[(combo, pat)] = [n, succ, succ]

    if not per_combo:
        return None
    return {"tag": "baseline", "per_pat": per_pat, "per_combo": per_combo, "no_checker": True}


def _majority_wrong_for_keys(csv_path: Path, combo_keys: list[str]) -> dict[str, str]:
    if not csv_path.is_file():
        return {}
    key_set = set(combo_keys)
    acc = {k: {} for k in combo_keys}
    dtypes = {
        "variant": "string",
        "combo_key": "string",
        "pred_state": "string",
        "state_correct": "string",
    }
    try:
        reader = pd.read_csv(
            csv_path,
            usecols=list(dtypes),
            dtype=dtypes,
            chunksize=200_000,
            low_memory=False,
        )
    except ValueError:
        return {}

    for chunk in reader:
        mask = (
            (chunk["variant"] == "attack")
            & (chunk["combo_key"].isin(key_set))
            & (~chunk["state_correct"].str.lower().isin(["true", "1"]))
        )
        sub = chunk.loc[mask, ["combo_key", "pred_state"]].dropna()
        if sub.empty:
            continue
        for (ck, ps), cnt in sub.value_counts().items():
            ck, ps = str(ck), str(ps)
            if ck in acc and ps:
                acc[ck][ps] = acc[ck].get(ps, 0) + int(cnt)

    return {ck: max(counts, key=counts.get) for ck, counts in acc.items() if counts}


_example_keys = unique_example_cases()
_cons_filter = set(_example_keys)
two_stats = {}

cons_dir = Path(other_spec["cons_dir"])
details = cons_dir / "attack" / "details.jsonl"
if not details.is_file():
    raise FileNotFoundError(f"No attack/details.jsonl at {details}")

per_pat, per_combo, wrong_counts, n_used = _stream_combined_with_majority(
    details, _cons_filter
)
two_stats[other_alias] = {
    "tag": other_alias,
    "per_pat": per_pat,
    "per_combo": per_combo,
    "wrong_counts": wrong_counts,
    "cons_dir": cons_dir,
}

_base = _baseline_stats_from_ranking(_cons_filter)
if _base is None:
    raise RuntimeError("Could not load baseline attack_ranking.")

baseline_dir = Path(MODEL_SOURCES["baseline"]["dir"])
baseline_tag = MODEL_SOURCES["baseline"].get("tag", "baseline")
baseline_csv = baseline_dir / f"results_{baseline_tag}.csv"
_base["wrong_by_combo"] = _majority_wrong_for_keys(baseline_csv, _example_keys)
two_stats["baseline"] = _base

_laser_groups = group_cases_by_laser(_example_keys)


def _max_combined_rate_with_state(alias: str, combo: str):
    st = two_stats[alias]

    if st.get("no_checker"):
        best = None
        for (c, _p), (n, comb, _flip) in st["per_pat"].items():
            if c != combo or n == 0:
                continue
            r = 100.0 * comb / n
            if best is None or r > best:
                best = r
        return best, st.get("wrong_by_combo", {}).get(combo)

    best = None
    best_state = None
    for (c, p), (n, comb, _flip) in st["per_pat"].items():
        if c != combo or n == 0:
            continue
        r = 100.0 * comb / n
        if best is None or r > best:
            best = r
            counts = st["wrong_counts"].get((c, p), {})
            best_state = max(counts, key=counts.get) if counts else None
    return best, best_state


def _pred_label_text(wstate):
    short = WRONG_STATE_SHORT.get(
        str(wstate), str(wstate) if pd.notna(wstate) else None
    )
    return f"{short}" if short else None


def _annotate_pred_inside(ax, bar, y_bar, txt, ymax, annot_size=ANNOT_SIZE):
    if not txt or y_bar <= 0:
        return
    x = bar.get_x() + bar.get_width() / 2
    if y_bar >= 18:
        y = min(y_bar - 2.0, ymax - 2.5)
        va = "top"
        color = "white"
    else:
        y = min(y_bar + 1.2, ymax - 2.5)
        va = "bottom"
        color = "black"
    ax.text(
        x,
        y,
        txt,
        ha="center",
        va=va,
        fontsize=annot_size,
        fontweight="bold",
        color=color,
        clip_on=True,
        zorder=5,
    )


def _format_attack_axis_label(combo_key: str) -> str:
    label = label_attack_axis(combo_key)
    label = label.replace("benign ", "").replace(" benign", "").strip()
    parsed = parse_combo_key(combo_key)
    if parsed and parsed.get("laser") == "ir":
        label = label.replace("(top)", "(top injection)")
        label = label.replace("(bottom)", "(bottom injection)")
        label = label.replace("(left)", "(left injection)")
        label = label.replace("(right)", "(right injection)")
        label = label.replace("(middle)", "(middle injection)")
    return label


def _plot_combined_axis(
    ax,
    laser: str,
    *,
    keys_override: list[str] | None = None,
    ymax_override: float | None = None,
):
    keys = keys_override if keys_override is not None else _laser_groups.get(laser, [])
    keys_present = [
        k for k in keys if any(two_stats[a]["per_combo"].get(k) for a in TWO_ALIASES)
    ]
    if not keys_present:
        ax.set_visible(False)
        return

    n_models = len(TWO_ALIASES)
    n_ex = len(keys_present)
    x_ex = np.arange(n_ex)
    bar_w = min(0.85 / max(n_models, 1), 0.32)
    offsets = (np.arange(n_models) - (n_models - 1) / 2.0) * bar_w

    combined_ceiling = max(
        [
            (_max_combined_rate_with_state(a, k)[0] or 0.0)
            for a in TWO_ALIASES
            for k in keys_present
        ]
        + [1.0]
    )
    ymax = (
        ymax_override
        if ymax_override is not None
        else min(100, combined_ceiling * 1.18 + 6)
    )

    for i, alias in enumerate(TWO_ALIASES):
        color = TWO_COLORS[alias]
        combined_vals = []
        pred_texts = []
        for k in keys_present:
            cv, cw = _max_combined_rate_with_state(alias, k)
            combined_vals.append(0.0 if cv is None else float(cv))
            pred_texts.append(_pred_label_text(cw))

        bars = ax.bar(
            x_ex + offsets[i],
            combined_vals,
            bar_w,
            color=color,
            alpha=1.0,
            edgecolor="black",
            linewidth=0.8,
            zorder=2,
        )
        for bar, cv, txt in zip(bars, combined_vals, pred_texts):
            _annotate_pred_inside(ax, bar, cv, txt, ymax)

    ax.set_ylabel("Attack success (%)", fontsize=AXIS_LABEL_SIZE)
    ax.set_xlabel("Traffic Light Status", fontsize=AXIS_LABEL_SIZE)
    ax.set_title(DISPLAY_TITLES.get(laser, laser), fontsize=TITLE_SIZE, fontweight="bold")
    ax.set_xticks(x_ex)
    ax.set_xticklabels(
        [_format_attack_axis_label(k) for k in keys_present],
        rotation=20,
        ha="right",
        fontsize=TICK_LABEL_SIZE,
    )
    ax.tick_params(axis="y", labelsize=TICK_LABEL_SIZE)
    ax.set_ylim(0, ymax)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(
        handles=[
            Patch(facecolor=TWO_COLORS["baseline"], edgecolor="black", label="CLIP"),
            Patch(
                facecolor=TWO_COLORS[other_alias],
                edgecolor="black",
                linewidth=0.8,
                alpha=1.0,
                label="CLIP + PRINGLE",
            ),
        ],
        loc="upper left",
        fontsize=LEGEND_SIZE,
        frameon=True,
        fancybox=False,
        edgecolor="0.7",
        framealpha=0.92,
    )


def _shared_ymax_for_lasers(lasers: list[str], laser_groups: dict | None = None) -> float:
    groups = laser_groups if laser_groups is not None else _laser_groups
    rates = []
    for laser in lasers:
        for k in groups.get(laser, []):
            if not any(two_stats[a]["per_combo"].get(k) for a in TWO_ALIASES):
                continue
            for a in TWO_ALIASES:
                r, _ = _max_combined_rate_with_state(a, k)
                rates.append(0.0 if r is None else float(r))
    ceiling = max(rates + [1.0])
    return min(100.0, ceiling * 1.18 + 6.0)


In [ ]:
_EXCLUDE_BENIGN = {"off", "red_yellow"}


def _filter_keys_no_off_ry(keys: list[str]) -> list[str]:
    out = []
    for k in keys:
        parsed = parse_combo_key(k)
        if not parsed:
            continue
        if parsed.get("benign") in _EXCLUDE_BENIGN:
            continue
        out.append(k)
    return out


_laser_groups_no_off_ry = {
    laser: _filter_keys_no_off_ry(keys) for laser, keys in _laser_groups.items()
}
_SHARED_YMAX_NO_OFF_RY = _shared_ymax_for_lasers(
    ["green", "red", "ir"], laser_groups=_laser_groups_no_off_ry
)
_IR_YMAX = 30

fig = plt.figure(figsize=(15, 10))
axs = fig.subplot_mosaic(
    [
        ["green", "red"],
        ["ir", "ir"],
    ],
    gridspec_kw={"height_ratios": [1.0, 1.05]},
)

_plot_combined_axis(
    axs["green"],
    "green",
    keys_override=_laser_groups_no_off_ry.get("green", []),
    ymax_override=_SHARED_YMAX_NO_OFF_RY,
)
_plot_combined_axis(
    axs["red"],
    "red",
    keys_override=_laser_groups_no_off_ry.get("red", []),
    ymax_override=_SHARED_YMAX_NO_OFF_RY,
)
_plot_combined_axis(
    axs["ir"],
    "ir",
    keys_override=_laser_groups_no_off_ry.get("ir", []),
    ymax_override=_IR_YMAX,
)

fig.suptitle(
    "Attack success rate: CLIP vs. CLIP + PRINGLE (off / red+yellow removed)",
    fontsize=SUPTITLE_SIZE,
    fontweight="bold",
    y=0.98,
)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()
